# 19 · Optimiseurs — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch19_optimiseurs/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 19.Q1 à 19.Q12, 19.R1 à 19.R3, 19.1 à 19.6, 19.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 19.14 à 19.19 | un pas constant sur une vallée, puis ta librairie optim.py : plannings, SGD, Momentum et Nesterov, Adagrad, RMSprop et Adadelta, Adam et AdamW, warmup et cosinus, clipping (*prochaine session de génération*) | | | |
| B | 19.20 à 19.22 | trop de momentum, les figures du chapitre, un Adam qui diverge (*prochaine session de génération*) | | | |
| C | 19.23 à 19.25 | lire des courbes de loss d'optimiseurs, les solveurs de MLPClassifier, batch contre stochastique contre mini-batch sur les lunes (*prochaine session de génération*) | | | |
| D | 19.26 à 19.28 | une course d'optimiseurs sur la fonction de Rosenbrock, huit optimiseurs sur MNIST, le défi des 98 % en cinq epochs (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Calculer à la main les premiers pas d'un planning, du momentum, de Nesterov, d'Adagrad, de RMSprop et d'Adam.
- Écrire et vérifier contre `torch.optim` les optimiseurs et les plannings de `mylearn.optim`.
- Comparer des optimiseurs sur des vallées, sur la fonction de Rosenbrock et sur MNIST, et défendre un choix.

**Rappel express** (conventions de PyTorch, $g$ le gradient, $t$ compté à partir de 1). Momentum : $b = \mu b + g$ ($b_1 = g_1$), $w \leftarrow w - \eta b$ ; Nesterov : $w \leftarrow w - \eta(g + \mu b)$ ; Adagrad : $s \leftarrow s + g^2$, $w \leftarrow w - \eta g/(\sqrt{s} + \epsilon)$ ; RMSprop : $v \leftarrow \alpha v + (1 - \alpha) g^2$, $w \leftarrow w - \eta g/(\sqrt{v} + \epsilon)$ ; Adam : $m \leftarrow \beta_1 m + (1 - \beta_1) g$, $v \leftarrow \beta_2 v + (1 - \beta_2) g^2$, $w \leftarrow w - \eta\,\frac{m/(1 - \beta_1^t)}{\sqrt{v/(1 - \beta_2^t)} + \epsilon}$ ; AdamW : $w \leftarrow w(1 - \eta\lambda)$ puis le pas d'Adam. Plannings : $\eta_0\gamma^k$, $\eta_0\gamma^{\lfloor k/s \rfloor}$, warmup $\eta_{\max} t/T_w$ puis cosinus.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 19.1 à 19.6 et 🧮 19.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une forme : entre guillemets (`"(3, 5)"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 19.7 et 19.8, la réflexion (🗣️ 19.10, ⚖️ 19.11, 📄 19.12, 🛠️ 19.13) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math


def momentum_torch_192(lrs, mu=0.9, w=1.0):
    """PyTorch momentum on L(w) = w**2 (gradient 2w): b1 = g1, b = mu b + g, w -= lr * b."""
    b = None
    for lr in lrs:
        g = 2 * w
        b = g if b is None else mu * b + g
        w = w - lr * b
    return w


def momentum_book_192(lrs, gamma=0.9, w=1.0):
    """The book's momentum: delta = gamma * delta - lr * g, w += delta."""
    delta = 0.0
    for lr in lrs:
        delta = gamma * delta - lr * 2 * w
        w = w + delta
    return w


def nesterov_book_193(steps, lr=0.1, mu=0.9, w=1.0):
    """Look-ahead Nesterov (the book, Sutskever et al.): returns the list of (look-ahead, gradient, w, v)."""
    v, out = 0.0, []
    for _ in range(steps):
        look = w + mu * v
        g = 2 * look
        v = mu * v - lr * g
        w = w + v
        out.append((look, g, w, v))
    return out


def nesterov_torch_193(steps, lr=0.1, mu=0.9, p=1.0):
    """PyTorch's Nesterov: gradient at the current point, p -= lr * (g + mu * b)."""
    b, out = None, []
    for _ in range(steps):
        g = 2 * p
        b = g if b is None else mu * b + g
        p = p - lr * (g + mu * b)
        out.append(p)
    return out


GRADS_194 = [3.0, -1.0, 2.0]


def adagrad_194(lr=0.1):
    s, w, out = 0.0, 0.0, []
    for g in GRADS_194:
        s += g * g
        w -= lr * g / math.sqrt(s)
        out.append((s, w))
    return out


def rmsprop_194(lr=0.1, alpha=0.9):
    v, w, out = 0.0, 0.0, []
    for g in GRADS_194:
        v = alpha * v + (1 - alpha) * g * g
        w -= lr * g / math.sqrt(v)
        out.append((v, w))
    return out


def adam_195(grads, lr=0.01, b1=0.9, b2=0.999, w=1.0):
    """Adam with eps = 0: returns the list of (m, v, m_hat, v_hat, w) after each step."""
    m = v = 0.0
    out = []
    for t, g in enumerate(grads, start=1):
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g * g
        m_hat, v_hat = m / (1 - b1**t), v / (1 - b2**t)
        w -= lr * m_hat / math.sqrt(v_hat)
        out.append((m, v, m_hat, v_hat, w))
    return out


SIZES_199 = [3072, 512, 256, 10]
P_199 = sum(a * b + b for a, b in zip(SIZES_199[:-1], SIZES_199[1:]))
W_199 = sum(a * b for a, b in zip(SIZES_199[:-1], SIZES_199[1:]))      # weights only, without the biases

**19.Q1 — Qu'appelle-t-on un optimiseur ?**

In [3]:
wb.record("19.Q1a", False, mistakes={"relis le chapeau de la fiche : qui calcule les gradients ?": True})
wb.record("19.Q1b", "B", mistakes={"les données ont déjà servi à la passe avant et à la passe arrière": "A",
          "les labels servent à calculer la loss, avant la passe arrière": "C",
          "l'architecture ne change pas d'un pas à l'autre : que faut-il à une règle comme w ← w − η g ?": "D"})
wb.record("19.Q1c", True, mistakes={"au même point, de quoi dépend le gradient : du réseau, des données et des poids, ou de l'optimiseur ?": False})
wb.record("19.Q1d", "B", mistakes={"backward() calcule les gradients, il ne touche pas aux poids": "A",
          "zero_grad() remet les gradients à zéro avant la passe arrière suivante": "C",
          "eval() change le mode du modèle (dropout, batchnorm), pas ses poids": "D"})

WB_ANSWER {"hash": "8e4e3e8b3023524cc7069beb0ba575f7bb1838c3a42e4e606eb7574663e415f6", "id": "19.Q1a", "kind": "bool", "mistakes": {"47158cae042a430056aa8b14d934052355b69e8e9876132bf85f49ce2acedf59": "relis le chapeau de la fiche : qui calcule les gradients ?"}}
📝 Ex 19.Q1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "270ec520b8bbcb4b205360b39a373cb9578e5ba58c5e859c33f05f921d93758a", "id": "19.Q1b", "kind": "str", "mistakes": {"2a7b83deac4658b2b0abee5775afe123e5c2149d07068b6581347331983f6a07": "les labels servent à calculer la loss, avant la passe arrière", "4e3c944de5cad2dc11ad52696f516201b1dda62ae9b4a0b7fb85d3bc7062aefb": "les données ont déjà servi à la passe avant et à la passe arrière", "898aea6d49fc2aac716852711aa233525adbe79d1a4b82a11ce99154c520e966": "l'architecture ne change pas d'un pas à l'autre : que faut-il à une règle comme w ← w − η g ?"}}
📝 Ex 19.Q1b : réponse enregistrée (str).
WB_ANSWER {"hash": "0070ef33cdd0eedfbbf665dc345642911c622f03c57ea507a149304ec120f9b2", 

**19.Q2 — Minimum, maximum, plateau, selle : où le gradient s'annule-t-il ?**

In [4]:
wb.record("19.Q2a", "A", mistakes={"x² + 3y² : monte-t-il ou descend-il quand on s'éloigne de 0 ?": "B",
          "les deux coefficients ont-ils des signes contraires ?": "C",
          "la fonction n'est pas constante autour de 0": "D"})
wb.record("19.Q2b", "C", mistakes={"regarde le signe du coefficient de y²": "A", "regarde le signe du coefficient de x²": "B",
          "cette fonction n'est pas constante autour de 0": "D"})
wb.record("19.Q2c", "B", mistakes={"regarde le signe des deux coefficients": "A",
          "les deux coefficients ont-ils des signes contraires ?": "C",
          "cette fonction-là n'est pas constante autour de 0": "D"})
wb.record("19.Q2d", "D", mistakes={"une fonction constante monte-t-elle quand on s'éloigne du point ?": "A",
          "une fonction constante descend-elle quand on s'éloigne du point ?": "B",
          "une fonction constante a-t-elle des directions qui montent et d'autres qui descendent ?": "C"})
wb.record("19.Q2e", True, mistakes={"écris le pas η g avec g = 0 : les poids bougent-ils ?": False})

WB_ANSWER {"hash": "229609243f70928c22436772f8ee2fac8aed1272f2b083d5128f9a9479f9defd", "id": "19.Q2a", "kind": "str", "mistakes": {"4603f534a5e38423337b874e09b5b247a7f1aaf12a924bc928c547fbdcf7baee": "x² + 3y² : monte-t-il ou descend-il quand on s'éloigne de 0 ?", "5cff3b5c18074dabfcdb7301e1ba0be11cfb76d03a8f772e55dbb26342fb63f0": "les deux coefficients ont-ils des signes contraires ?", "7187c8d61b7c54c80a8464ab848900139b308bc69bc23a33639a3439b9ee18f4": "la fonction n'est pas constante autour de 0"}}
📝 Ex 19.Q2a : réponse enregistrée (str).
WB_ANSWER {"hash": "76bd42114ae6ca67f691cb26cd2a3d4a5cf0ffffc3f92c8b3d49485d66cebb6a", "id": "19.Q2b", "kind": "str", "mistakes": {"10f4d0c4c159a0c1f1d1b2e3ad2bfa5145c85384a876b6155ca54a58b6385dc4": "regarde le signe du coefficient de x²", "514d38566617ca53b9b0f8a11fad63e4c8bca812cc60200c8f637690c93add89": "cette fonction n'est pas constante autour de 0", "b271ff4e2eec7b6d82ec6abc4840afd518a933f025b49d418d2f88dedfe4e35f": "regarde le signe du coeffic

**19.Q3 — Minima locaux ou points selles : le vrai piège en grande dimension**

In [5]:
wb.record("19.Q3a", "B", mistakes={"relis la fin du §19.2.1 : la thèse de Dauphin et coll. porte sur un autre type de point": "A",
          "les maxima sont aussi rares que les minima dans ce raisonnement": "C",
          "rien à voir avec les arrondis : relis la fin du §19.2.1": "D"})
wb.record("19.Q3b", 2**10, fractional="« 1 sur combien » : un nombre entier (pas une probabilité)",
          mistakes={"une chance sur deux pour CHAQUE direction : on multiplie les probabilités": 10,
          "une chance sur deux pour chacune des 10 directions, toutes doivent monter": 512,
          "on multiplie les probabilités, on ne les additionne pas": 20,
          "une chance sur deux par direction, pas une sur dix": 100,
          "compte toutes les configurations de signes, celle du minimum comprise": 1023})
wb.record("19.Q3c", True, mistakes={"que devient la probabilité d'un minimum quand d grandit ? Et celle d'un maximum ?": False})
wb.record("19.Q3d", True, mistakes={"une selle a des directions qui descendent : une petite poussée au hasard peut-elle en trouver une ?": False})

WB_ANSWER {"hash": "430c4b1906680a39df84540e195b7b2886c178758d61b061ad51e85364e33399", "id": "19.Q3a", "kind": "str", "mistakes": {"1c513f5448c37a22de96565f001ed69789ca7332566b3c20bb7e7f4106b28a61": "les maxima sont aussi rares que les minima dans ce raisonnement", "202a7e18ead05ae6ac48c576613e93aaae08d8a4aed4accab6de9d90695b1ecc": "relis la fin du §19.2.1 : la thèse de Dauphin et coll. porte sur un autre type de point", "7edc7bf4981c99bccf3f5a51deb6461dab5e709a816e0c08d8dded73bde9a711": "rien à voir avec les arrondis : relis la fin du §19.2.1"}}
📝 Ex 19.Q3a : réponse enregistrée (str).
WB_ANSWER {"fractional": "« 1 sur combien » : un nombre entier (pas une probabilité)", "hash": "3e8ae81b869e52aa5d6dcf6443244f65d1662017da054d57ffba6155094c1afb", "id": "19.Q3b", "kind": "int", "magnitude": 3, "mistakes": {"004a9ec779fe2d84e21f5a411a0356ef36aa8b13aa644ee0725c5b5cbc7ed173": "une chance sur deux par direction, pas une sur dix", "0f2af57e906fd83d9c0235469733071a28f0cd647b003694700cf4024caf

**19.Q4 — Pourquoi lisser une erreur de comptage ?**

In [6]:
wb.record("19.Q4a", True, mistakes={"le nombre d'erreurs change-t-il quand le seuil se déplace entre deux points ?": False})
wb.record("19.Q4b", True, mistakes={"quelle est la pente d'une fonction constante ?": False})
wb.record("19.Q4c", "B", mistakes={"lisser ne rend pas le calcul plus rapide : que manque-t-il à la descente sur un escalier ?": "A",
          "un lissage ne supprime pas forcément les minima locaux": "C",
          "le problème n'est pas la valeur au minimum, c'est la pente ailleurs": "D"})
wb.record("19.Q4d", "B", mistakes={"l'accuracy compte des erreurs : elle est en escalier": "A",
          "un nombre de faux positifs est un comptage": "C",
          "le F1 se calcule à partir de comptages": "D"})

WB_ANSWER {"hash": "ff44dbd11cb1353098bd0c0eedbaa720d81b2ea6bd18b4c6e69d9981afd32af0", "id": "19.Q4a", "kind": "bool", "mistakes": {"125ca9b012abc38356acf50b0c252282d04b1b0bc2bec8c8ce2c39395d388107": "le nombre d'erreurs change-t-il quand le seuil se déplace entre deux points ?"}}
📝 Ex 19.Q4a : réponse enregistrée (bool).
WB_ANSWER {"hash": "b39a3fd3b69b0d6560868e5c0c6578c6391375d76b68fa7649e5f34507f5d1ba", "id": "19.Q4b", "kind": "bool", "mistakes": {"005d35873b32c8b3884d02b218661c1d7b9fde16347619d1ad769fec4a502df6": "quelle est la pente d'une fonction constante ?"}}
📝 Ex 19.Q4b : réponse enregistrée (bool).
WB_ANSWER {"hash": "1ebbc4435f15a2f521ef8c27782f049c4d98e0f065147840ae9f8759bb2e9a3c", "id": "19.Q4c", "kind": "str", "mistakes": {"7b18c996182249d902502f33b07714837c72ff8952084152a76fa2384ff4d20e": "lisser ne rend pas le calcul plus rapide : que manque-t-il à la descente sur un escalier ?", "9cd2ccff70a77093823a62b66b4fe7e5d3d63caa20d06251fdff4dbb574fae56": "un lissage ne supprim

**19.Q5 — Pas constant : pourquoi rebondit-on au fond de la vallée ?**

In [7]:
wb.record("19.Q5a", 1 - 10 * 0.15, decimals=2, mistakes={"regarde le signe de 1 − 10η": 0.5,
          "L(w) = 5w² : sa dérivée vaut 10w, la courbure est 10": 0.25,
          "le facteur dépend de la courbure, pas seulement de η": 0.85})
wb.record("19.Q5b", True, mistakes={"le facteur de a) est-il positif ou négatif ?": False})
wb.record("19.Q5c", "D", mistakes={"calcule 1 − 10 × 0,25 : est-il positif ?": "A",
          "compare la valeur absolue du facteur à 1": "B",
          "le facteur vaut-il exactement −1 ?": "C"})
wb.record("19.Q5d", "C", mistakes={"calcule 1 − 10 × 0,2 : est-il positif ?": "A",
          "un facteur de module exactement 1 rapproche-t-il du fond ?": "B",
          "un facteur de module exactement 1 fait-il grandir l'écart ?": "D"})
wb.record("19.Q5e", True, mistakes={"pour quels η le module du facteur est-il strictement inférieur à 1 ?": False})

WB_ANSWER {"decimals": 2, "hash": "79b60a1c291906bbe029821a6c19096fbad7a69f1f2b1097ff5aa5123fdcd66c", "hash_coarse": "ceb6d3556f2aacbdb0ce8c0220af1f0c88b5bc7070946195c094c10905fad0cf", "id": "19.Q5a", "kind": "float", "magnitude": -1, "mistakes": {"424b7a22d67843e53bea2d0bca2922f4c17ae524ab93e0c2d9804c8b73797e23": "regarde le signe de 1 − 10η", "53c9983ced13a5f1bbf772854231e40ac4eeb356af4ef621873ba08665eaee56": "le facteur dépend de la courbure, pas seulement de η", "c6515efc39cd38d67302d61d64612728ef14aca66b3d522a53a8e2834727932b": "L(w) = 5w² : sa dérivée vaut 10w, la courbure est 10"}, "sign": -1}
📝 Ex 19.Q5a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "c7bce4fd813ce5ac15250efc9284e7b90ffe897abfb24c8d2c2478f6aee4b42a", "id": "19.Q5b", "kind": "bool", "mistakes": {"3e3bb3ee3c4b99983f9a3db3f6469ab3683b2d17f8d162be5e1ac0344b96c24b": "le facteur de a) est-il positif ou négatif ?"}}
📝 Ex 19.Q5b : réponse enregistrée (bool).
WB_ANSWER {"hash": "44074624d7f3d40b4aec693

**19.Q6 — Décroissance exponentielle : que vaut η après 50 pas ?**

In [8]:
wb.record("19.Q6a", 0.1 * 0.99**50, decimals=4, mistakes={"50 multiplications : la puissance est 50": 0.1 * 0.99**49,
          "50 multiplications, pas 51": 0.1 * 0.99**51,
          "une décroissance exponentielle multiplie, elle ne soustrait pas": 0.05,
          "n'oublie pas η0 = 0,1": 0.99**50})
wb.record("19.Q6b", math.ceil(math.log(0.5) / math.log(0.99)), fractional="un nombre de multiplications : un entier, le premier qui convient",
          mistakes={"après ce nombre de multiplications, η est-il déjà sous la moitié de η0 ?": 68,
          "le PREMIER nombre de multiplications qui convient": 70,
          "relis a) : après 50 multiplications, est-on déjà sous la moitié ?": 50})
wb.record("19.Q6c", "B", mistakes={"calcule 0,8 puissance 10, puis son inverse": "A", "calcule 0,8 puissance 10 : est-ce si petit ?": "C",
          "0,8 puissance 10 n'est pas si petit : refais le calcul": "D"})
wb.record("19.Q6d", True, mistakes={"une série géométrique de raison strictement inférieure à 1 converge-t-elle ?": False})
wb.record("19.Q6e", 0.1 / (1 - 0.99), decimals=2, mistakes={"n'oublie pas η0 = 0,1": 100.0,
          "divise par 1 − γ, pas par γ": 0.1 / 0.99,
          "la somme d'une série géométrique divise par 1 − γ, elle ne multiplie pas": 0.001,
          "η0 = 0,1 multiplie la somme : n'oublie pas sa valeur": 1000.0})

WB_ANSWER {"decimals": 4, "hash": "475ac43e73c72595459e1e86587b2397c3c55d9c17d27dcdfafecf8953c5c9c2", "hash_coarse": "b990f544e2748f4cdf69d7f02eafd5470dff56d791eabbec1f72d9273cbc310b", "id": "19.Q6a", "kind": "float", "magnitude": -2, "mistakes": {"2b4ce648b148b5ea3cd750ce4d4f126e29763b5680b25e652c0d232c0e57eae9": "50 multiplications : la puissance est 50", "89278c974b7d04d474fe8f5b057db6094209b7315711dd55f2922aa2938980f7": "50 multiplications, pas 51", "b2a7a43b4346e780b7189f93e986b4ecbd545a1cafed1c7a7db6b55a5fcf6428": "une décroissance exponentielle multiplie, elle ne soustrait pas", "f1049c13ca7b5952221a930d125733312e53cf8746c03c088b62ca51e0f58a36": "n'oublie pas η0 = 0,1"}, "sign": 1}
📝 Ex 19.Q6a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"fractional": "un nombre de multiplications : un entier, le premier qui convient", "hash": "68dc8b8e0627da09f85716d01d7e37bd2737a4083d9d699261a2e47a2016ba0b", "id": "19.Q6b", "kind": "int", "magnitude": 1, "mistakes": {"0b143a6d23339

**19.Q7 — Plannings : exponentiel, retardé, par paliers, sur l'erreur, bold driver**

In [9]:
wb.record("19.Q7a", "C", mistakes={"η baisse-t-il à chaque epoch, ou toutes les 10 ?": "A", "y a-t-il un délai au départ ?": "B",
          "ici, c'est le calendrier qui décide, pas la loss": "D", "η ne monte jamais dans cette règle": "E"})
wb.record("19.Q7b", "B", mistakes={"que se passe-t-il pendant les 5 premières epochs ?": "A",
          "η baisse-t-il par marches, ou à chaque epoch une fois lancé ?": "C",
          "ici, c'est le calendrier qui décide : la loss n'intervient pas": "D",
          "η ne monte jamais dans cette règle-là": "E"})
wb.record("19.Q7c", "D", mistakes={"ici, c'est la loss qui décide, pas un calendrier": "A",
          "ici, c'est la loss qui décide, pas un délai": "B",
          "pas de calendrier fixe dans cette règle": "C",
          "η peut-il monter dans cette règle ?": "E"})
wb.record("19.Q7d", "E", mistakes={"η augmente parfois : un planning exponentiel le fait-il ?": "A",
          "η augmente parfois : un planning retardé le fait-il ?": "B",
          "η augmente parfois : un planning par paliers le fait-il ?": "C",
          "η augmente parfois : le planning sur l'erreur le fait-il ?": "D"})
wb.record("19.Q7e", "C", mistakes={"StepLR suit un calendrier fixe": "A", "ExponentialLR multiplie à chaque epoch, quoi qu'il arrive": "B",
          "CosineAnnealingLR suit une courbe fixée d'avance": "D"})
wb.record("19.Q7f", True, mistakes={"relis l'encadré 🕰️ du §19.3.3 : en quoi se comptent les plannings d'aujourd'hui ?": False})

WB_ANSWER {"hash": "5dc69c6bd37be5adcb6085ebe5a06bc5b65e3ac656d180c11ad04e61387e66fd", "id": "19.Q7a", "kind": "str", "mistakes": {"0e296e5a78c14870a336a121cee55664c6c1d2e845b89a7f30fe8c2fcf6c524b": "η baisse-t-il à chaque epoch, ou toutes les 10 ?", "1e21de552002911d0dfd7ed5d6b12e8c1bb25caa17105212e53e37be057faee4": "ici, c'est le calendrier qui décide, pas la loss", "8489c07a16b36aacb3bd4111f6b8440fcb30e9475a17a85ba553a1c43089e37c": "y a-t-il un délai au départ ?", "ac79b8eddc4eb22466f6c4199a44e1e5a6cbc4084ced7e4a521bf98b275e357a": "η ne monte jamais dans cette règle"}}
📝 Ex 19.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "1c17fc1351203cbcd5be520d75ddb09e40d707735c3e068aacb829c92c625826", "id": "19.Q7b", "kind": "str", "mistakes": {"0fc5d64eec8345951b6a380fd53b60755d4799b284a3deaa9083c497606d9ed9": "η ne monte jamais dans cette règle-là", "549141b7e625ad29c86c05eb3da2a3f53fd04320716b754bcb17450d15cb6d57": "que se passe-t-il pendant les 5 premières epochs ?", "b8f64b522aa6f49d3

**19.Q8 — Batch, stochastique, mini-batch : compter les mises à jour**

In [10]:
wb.record("19.Q8a", 1, mistakes={"le batch, c'est tout le dataset : combien de mises à jour par epoch ?": 1200,
          "chaque epoch fait au moins une mise à jour": 0})
wb.record("19.Q8b", 1200, mistakes={"un exemple par mise à jour : combien de mises à jour pour 1 200 exemples ?": 1})
wb.record("19.Q8c", math.ceil(1200 / 64), fractional="un nombre de mises à jour : un entier (le dernier mini-batch, incomplet, compte aussi)",
          mistakes={"le dernier mini-batch, incomplet, compte aussi": 18,
          "combien de mini-batches de 64 dans 1 200 exemples ?": 64})
wb.record("19.Q8d", 50 * math.ceil(1200 / 64), mistakes={"le dernier mini-batch incomplet compte, à chaque epoch": 900,
          "ce sont des mini-batches de 64, pas des exemples isolés": 60000})
wb.record("19.Q8e", "B", mistakes={"la descente par batch a besoin de tous les exemples à chaque mise à jour": "A",
          "l'une des deux a besoin de garder tous les exemples": "C",
          "l'une des deux traite chaque exemple dès qu'il arrive": "D"})
wb.record("19.Q8f", False, mistakes={"une epoch de SGD et une epoch de batch font-elles le même nombre de mises à jour ?": True})

WB_ANSWER {"hash": "3cab1106ec13a9b7bb0c8f56556cd2abf5329ea7c72257b59f6521a5f58d4b00", "id": "19.Q8a", "kind": "int", "magnitude": 0, "mistakes": {"82a86437ac0d1c7252b4119d8c751fc1b431cca09140a706ee34546f373375f3": "chaque epoch fait au moins une mise à jour", "cdc9abd47f6279fc6dfa7b0688546d5f65d9b541ad260e5ab6a2ac609d9ae121": "le batch, c'est tout le dataset : combien de mises à jour par epoch ?"}, "sign": 1}
📝 Ex 19.Q8a : réponse enregistrée (int).
WB_ANSWER {"hash": "3973dcb411ccfac6c9ce736b427a8ee26079e14c2e3de0181207b4ee15fdf23e", "id": "19.Q8b", "kind": "int", "magnitude": 3, "mistakes": {"68ac15d6c951b2b342dc29ba1b573c9730ddf404baeebbecc5d88f6d6062d887": "un exemple par mise à jour : combien de mises à jour pour 1 200 exemples ?"}, "sign": 1}
📝 Ex 19.Q8b : réponse enregistrée (int).
WB_ANSWER {"fractional": "un nombre de mises à jour : un entier (le dernier mini-batch, incomplet, compte aussi)", "hash": "390e12ffa9d9791ce5880a57b141345765a377d934d7a50f070849e41a052ec5", "id": "1

**19.Q9 — Momentum : que se passe-t-il si γ = 0 ? si γ = 1 ?**

In [11]:
wb.record("19.Q9a", "A", mistakes={"avec μ = 0, b = g : le pas est-il nul ?": "B", "Adam a deux moyennes mobiles": "C",
          "μ = 0 ne change pas la stabilité de la descente ordinaire": "D"})
wb.record("19.Q9b", 1 / (1 - 0.9), decimals=1, mistakes={"b1 = 1, b2 = 1 + 0,9, b3 = 1 + 0,9 + 0,81… : la somme commence à 1": 9.0,
          "la vitesse s'accumule : déroule b2 et b3": 1.0,
          "c'est le coefficient μ, pas la limite de b": 0.9})
wb.record("19.Q9c", "B", mistakes={"avec μ = 1, b_t = b_(t−1) + 1 : s'approche-t-il d'une limite ?": "A",
          "on ajoute 1 à chaque pas : est-ce une croissance exponentielle ?": "C",
          "b change-t-il d'un pas à l'autre ?": "D"})
wb.record("19.Q9d", "B", mistakes={"b_t = b_(t−1) + 0 : la vitesse diminue-t-elle ?": "A",
          "rien ne change le signe de la vitesse quand le gradient est nul": "C"})
wb.record("19.Q9e", True, mistakes={"relis l'histoire de la bille sur le plateau (§19.5.1)": False})

WB_ANSWER {"hash": "28639036e3d88b09bb0bceb9882350cd3be040cc1a0c56dce30b16b7ab587924", "id": "19.Q9a", "kind": "str", "mistakes": {"1ddfaca57981eea2b3d6eb376624e2034ab831e9b66eb86c21b5b3c8bc6f45df": "Adam a deux moyennes mobiles", "344638fdf0cf4c31097344f18ec92a5bf1f781c7d78a4dbc38c0fa63ce3e3a37": "μ = 0 ne change pas la stabilité de la descente ordinaire", "af1435255bde62f8dbcb546516cc379ec14543e4ac42981938b5b43ef4ea3f03": "avec μ = 0, b = g : le pas est-il nul ?"}}
📝 Ex 19.Q9a : réponse enregistrée (str).
WB_ANSWER {"decimals": 1, "hash": "6cccd891ef3d37bc341581038571787cf33a710ae588239b45b1fae19912186f", "hash_coarse": "5fb32706bb6022062a3ac8fdd1fbad4e3644758a345f189665d8ad167b973bad", "id": "19.Q9b", "kind": "float", "magnitude": 1, "mistakes": {"22dce543125e26deea42e7aef07d1667897b3ef9a0fa4c7485f106504a686336": "b1 = 1, b2 = 1 + 0,9, b3 = 1 + 0,9 + 0,81… : la somme commence à 1", "bd69dddb7da2cb1867259c12811ff79b9d0077f4f5bc078634941826c7263c3f": "c'est le coefficient μ, pas la li

**19.Q10 — Nesterov : regarder là où l'on va**

In [12]:
wb.record("19.Q10a", "B", mistakes={"relis la formule du livre : où évalue-t-on ∇L ?": "A",
          "Nesterov regarde vers l'avant, pas en arrière": "C", "un seul gradient par pas": "D"})
wb.record("19.Q10b", False, mistakes={"compte les hyperparamètres de la formule de Nesterov": True})
wb.record("19.Q10c", True, mistakes={"relis la phrase sur le changement de variable (§19.5.2)": False})
wb.record("19.Q10d", "B", mistakes={"de l'autre côté du fond, dans quel sens pointe le gradient ?": "A",
          "le point anticipé a-t-il le même gradient que le point courant ?": "C",
          "le gradient s'ajoute à l'élan, il ne le remplace pas": "D"})

WB_ANSWER {"hash": "1fefac544a5de1273e0e4bb1f418e01d04761a947e61a3b48eb508ee309dca48", "id": "19.Q10a", "kind": "str", "mistakes": {"53477c9c9be65d8db819608df8bb29387eaabec237b9222f10509ab54b155d17": "relis la formule du livre : où évalue-t-on ∇L ?", "8a25c36299d474c34ec8a0a7b10157d8b1d0ed1d195c9ad8b452ea87f1f088c7": "un seul gradient par pas", "8c84652ad1cb59511ce9e2b53a86197450ecf36dc00123ddc43e63c13636c1e8": "Nesterov regarde vers l'avant, pas en arrière"}}
📝 Ex 19.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "9be9da6af76a1f06fcd0387561523ac469995300f7de3adfab8a02dc17203280", "id": "19.Q10b", "kind": "bool", "mistakes": {"47966206ea5e20ab08e4743db69e67efcf8f48ab6a2754542b8e8f1dd4b7eaa5": "compte les hyperparamètres de la formule de Nesterov"}}
📝 Ex 19.Q10b : réponse enregistrée (bool).
WB_ANSWER {"hash": "181328715bfc9eb7a8ec51ac5159a1c70d0205bb2d3f189e65958d6ca6cef6c0", "id": "19.Q10c", "kind": "bool", "mistakes": {"2725ffba5cc433a016003b92cffdd305c53a6eebdfe894e15e9227a0cc

**19.Q11 — Adagrad, Adadelta, RMSprop : un learning rate par poids**

In [13]:
wb.record("19.Q11a", True, mistakes={"chaque poids a-t-il son propre accumulateur ?": False})
wb.record("19.Q11b", "A", mistakes={"relis la formule d'Adagrad : oublie-t-il ses anciens gradients ?": "B",
          "le learning rate d'Adagrad est fixe : d'où vient le rétrécissement ?": "C",
          "le carré efface le signe : relis ce qu'on ajoute à s": "D"})
wb.record("19.Q11c", 1 - 0.9, decimals=2, mistakes={"c'est le poids de l'ancienne moyenne": 0.9,
          "une part entre 0 et 1, sans pourcentage": 10.0})
wb.record("19.Q11d", "B", mistakes={"lr est le learning rate": "A", "momentum ajoute une vitesse au pas déjà normalisé": "C",
          "eps évite une division par zéro": "D"})
wb.record("19.Q11e", 0.99, decimals=2, mistakes={"c'est la valeur du cours de Hinton et du livre, pas le défaut de PyTorch": 0.9,
          "relis la valeur par défaut de alpha dans PyTorch": 0.999})
wb.record("19.Q11f", True, mistakes={"relis la phrase sur l'algorithme d'origine d'Adadelta (§19.5.4)": False})

WB_ANSWER {"hash": "946d7a19913671d68d2286a3def3d5783be313373987ca98eca17325571eddff", "id": "19.Q11a", "kind": "bool", "mistakes": {"2b7eb993c0c380784c22d96bf80412b55386229effd17802cd6e552b793f005b": "chaque poids a-t-il son propre accumulateur ?"}}
📝 Ex 19.Q11a : réponse enregistrée (bool).
WB_ANSWER {"hash": "c7f4e16762ffd674e2cb144b89c02b204f8bdd24c2487b138b98e6f9fc62f1fb", "id": "19.Q11b", "kind": "str", "mistakes": {"527fc2daa4ab8302e38e0c3186f6d000e8805d79885646540459d29f75b4beff": "le learning rate d'Adagrad est fixe : d'où vient le rétrécissement ?", "780a000b53e4fb04711282943be661e54719e7bb15a42af921ff1f5b3309240c": "le carré efface le signe : relis ce qu'on ajoute à s", "cae5a4d0c4bd18c62fb70fb45db9c763e431be53fdade47a749bc3271c34bd40": "relis la formule d'Adagrad : oublie-t-il ses anciens gradients ?"}}
📝 Ex 19.Q11b : réponse enregistrée (str).
WB_ANSWER {"decimals": 2, "hash": "2e65683065208145a885d3c53ec1b78913843951245b9ed04561b3603d16f53b", "hash_coarse": "044615e1fbb17

**19.Q12 — Adam, No Free Lunch et choix d'un optimiseur**

In [14]:
wb.record("19.Q12a", "A", mistakes={"relis les deux moyennes d'Adam et le rôle de chacune": "B",
          "le clipping et le warmup ne font pas partie d'Adam": "C",
          "ce sont des plannings, pas des parties d'Adam": "D"})
wb.record("19.Q12b", 0.9, decimals=3, mistakes={"relis les valeurs proposées par Kingma et Ba, et laquelle va avec β1": 0.999,
          "relis les valeurs proposées par Kingma et Ba": 0.99})
wb.record("19.Q12c", 0.999, decimals=3, mistakes={"relis les valeurs proposées par Kingma et Ba, et laquelle va avec β2": 0.9,
          "0,99 n'est pas le β2 de Kingma et Ba : relis leurs valeurs": 0.99})
wb.record("19.Q12d", False, mistakes={"relis l'encadré sur le No Free Lunch : en moyenne sur quoi ?": True})
wb.record("19.Q12e", "C", mistakes={"relis le bilan de la figure 19.49 : Adagrad met des milliers d'epochs": "A",
          "relis le bilan de la figure 19.49 : Adadelta n'est pas en tête": "B",
          "relis le bilan de la figure 19.49 : Adam arrive juste derrière": "D"})

WB_ANSWER {"hash": "37a5880baee67c4cefb21b4680854e8fe3957830e78740d5e565b01b96ce85f0", "id": "19.Q12a", "kind": "str", "mistakes": {"11087506becb3a7eb0061c263bf05db38870bec8b767f9005cb0f58a63a8fe83": "ce sont des plannings, pas des parties d'Adam", "172ab9f26a531b68ce8fd00641cac9c38c6c02e5a0bf99c2ff614879c3d4504c": "relis les deux moyennes d'Adam et le rôle de chacune", "802e3cd44e8bcb6e5d0b7fa136b8e0deae272a491a1cd519e71b893e6310f4c8": "le clipping et le warmup ne font pas partie d'Adam"}}
📝 Ex 19.Q12a : réponse enregistrée (str).
WB_ANSWER {"decimals": 3, "hash": "74ed4a1e8a7a348c6f24406511d93b0ee304e0c48bb5c38b30d0d639dd679680", "hash_coarse": "5ea0d04ba959b2b8ccbe94beb3b7b227574416940f1a4656888a30d56eda7146", "id": "19.Q12b", "kind": "float", "magnitude": -1, "mistakes": {"36750ee62e4383c38a947c7e35e9c175535eb11614a288b230de46029af16e9a": "relis les valeurs proposées par Kingma et Ba, et laquelle va avec β1", "b3e0714dcdbcb33b16d5459e436914e6e5ab09ba1a38f31dca8ab0f3a01519e1": "reli

**19.R1 — Du delta au gradient d'un poids**

In [15]:
wb.record("19.R1a", 0.4 * -2.5, decimals=2, mistakes={"multiplie a_i par δ_j, ne les additionne pas": 0.4 - 2.5,
          "a_i fois δ_j, pas δ_j divisé par a_i": -2.5 / 0.4})
wb.record("19.R1b", -0.05 * (0.4 * -2.5), decimals=3, mistakes={"w ← w − η ∂L/∂w : attention au double signe": -0.05,
          "η multiplie le gradient du poids, a_i δ_j, pas le delta seul": 0.125})
wb.record("19.R1c", "(64, 10)", mistakes={"le gradient a la forme du poids, (n_in, n_out) : relis la transposée": "(10, 64)",
          "on somme sur les exemples : le gradient a la forme du poids": "(32, 10)",
          "Aᵀ Δ : (64, 32) fois (32, 10)": "(32, 32)",
          "écris la forme entre parenthèses, comme dans l'exemple de format": "64, 10"})
wb.record("19.R1d", False, mistakes={"l'optimiseur reçoit le gradient : le calcule-t-il ?": True})

WB_ANSWER {"decimals": 2, "hash": "887c1efef57e1572c48884ba7047ae99074f4d36ac0a90eb9c8b653707188c00", "hash_coarse": "b94138e889c4d0228514fdd4dccf19901472a077c0e293679d19be73ebb3205b", "id": "19.R1a", "kind": "float", "magnitude": 0, "mistakes": {"1815bb136da04c359a1487d38f25c9416d0452e52d3c085d8e9da1a8645fb5c4": "multiplie a_i par δ_j, ne les additionne pas", "cae5b866119228a4db86ea8181a4d5b7c93d730deac22bd6e45423487573fc12": "a_i fois δ_j, pas δ_j divisé par a_i"}, "sign": -1}
📝 Ex 19.R1a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "6eb62b2c840391d1091af4809cd947d5274473430a2b8213ad7a649a387adef8", "hash_coarse": "d2e5719fff4cc56401340aa37a3d511da66e2bd7e98230e71cd7298494225838", "id": "19.R1b", "kind": "float", "magnitude": -2, "mistakes": {"0390e81f006975951e580d1db6a731b2279c035258e0201ec9584c61febcf2a6": "η multiplie le gradient du poids, a_i δ_j, pas le delta seul", "1b27589c787afd547224c3d44e146f18e03104062b143bdc145ac329922a7272": "w ← w − η

**19.R2 — Pourquoi l'initialisation He garde la variance stable**

In [16]:
wb.record("19.R2a", 2 / 200, decimals=3, mistakes={"He : 2 / n_in, pas 1 / n_in": 0.005, "relis ce qui est demandé : variance ou écart-type ?": 0.1,
          "c'est la formule de Glorot, avec n_out": 0.008})
wb.record("19.R2b", math.sqrt(2 / 200), decimals=3, mistakes={"relis ce qui est demandé ici : variance ou écart-type ?": 0.01,
          "He : variance 2 / n_in, puis la racine": math.sqrt(1 / 200)})
wb.record("19.R2c", 2 / (200 + 50), decimals=4, mistakes={"c'est He : Glorot fait intervenir n_out aussi": 0.01,
          "Glorot : 2 / (n_in + n_out), pas 1 / (n_in + n_out)": 0.004,
          "additionne n_in et n_out avant de diviser": 0.04})
wb.record("19.R2d", "A", mistakes={"la formule porte sur une seule couche": "B", "les biais partent de 0": "C",
          "la dérivée de ReLU vaut 0 ou 1": "D"})

WB_ANSWER {"decimals": 3, "hash": "dca6d79c86db056a4f5ee187ccf67461cc4a4465d4a11c52a6d01c5ecb9843e4", "hash_coarse": "37f544136302ac483bc326d4f1f20d826b4816736e57590df43b38e72038a450", "id": "19.R2a", "kind": "float", "magnitude": -2, "mistakes": {"273b3dbe9f84166d22e6a5a489c1a5574afd036fe9563b714048b26dd88a476d": "relis ce qui est demandé : variance ou écart-type ?", "6388652a90f9aca2cb5041e18fdc4ce93de05af0848b46c1024ea45415740b8e": "c'est la formule de Glorot, avec n_out", "77a0eb564c426d700b3205158cabc72acb81af4edbe89a79d9afda1cc78a4153": "He : 2 / n_in, pas 1 / n_in"}, "sign": 1}
📝 Ex 19.R2a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "9a98d883b582f740fa08825868fc28dcd2376c090a26d3cd918a352d47ce2afa", "hash_coarse": "d42f35088a4fd16c8f6e4aa27d60645af5f85e601158a0716a2a8474a06085bd", "id": "19.R2b", "kind": "float", "magnitude": -1, "mistakes": {"1447fcd5869e382b8d14945f842f6946042942e4a6aed7112d0da23a5de2d56d": "relis ce qui est demandé ici : va

**19.R3 — Standardiser les features, et l'effet sur la descente**

In [17]:
wb.record("19.R3a", round(1e6 / 0.25), mistakes={"le rapport de la plus grande à la plus petite : divise, ne multiplie pas": 250000,
          "le rapport des valeurs propres, pas de leurs racines": 2000})
wb.record("19.R3b", 2 / 1e6 / 1e-6, decimals=1, mistakes={"quelle courbure limite le learning rate : la plus grande ou la plus petite ?": 8000000.0,
          "la limite de convergence est 2 / λ_max : relis l'énoncé": 1.0,
          "en millionièmes : écris le nombre de millionièmes, pas la valeur": 0.000002})
wb.record("19.R3c", 1, mistakes={"après standardisation, que vaut la moyenne des carrés de chaque feature ?": 4000000,
          "un rapport de deux courbures positives peut-il valoir 0 ?": 0})
wb.record("19.R3d", True, mistakes={"relis l'encadré : diviser chaque gradient par sa taille habituelle compense-t-il les échelles ?": False})

WB_ANSWER {"hash": "2cd7a35d08c051dd71dfab9bf499ad9c504699e7d03d38d57e16c39b4f756e16", "id": "19.R3a", "kind": "int", "magnitude": 6, "mistakes": {"249b2e3a27b1763d6571426965abf69282933b4b61cab524ddaab97aa0fc205f": "le rapport des valeurs propres, pas de leurs racines", "b64c3d0537817839c9d50ec0883e497684447df1a22283b9002f14332db6d27e": "le rapport de la plus grande à la plus petite : divise, ne multiplie pas"}, "sign": 1}
📝 Ex 19.R3a : réponse enregistrée (int).
WB_ANSWER {"decimals": 1, "hash": "1deb6f6673b56e77e0e38bbf332be98cabb7d46f33567455f24813e789b89c84", "hash_coarse": "12db52727cfe122c9582fe0f5a60f783dda855525909b02cbc72aedddc4ff8be", "id": "19.R3b", "kind": "float", "magnitude": 0, "mistakes": {"25f278c92617d87e8aa13ea966a8963ba113297c0043722f0795df62a5c3773d": "la limite de convergence est 2 / λ_max : relis l'énoncé", "b356974385f252646ee422fb79fc3ade83e209edf138c71afdbeed09e45dd69b": "en millionièmes : écris le nombre de millionièmes, pas la valeur", "fd4d1e6cfde0a9b053872

**Ex 19.1 — Plannings exponentiel et par paliers à la main**

In [18]:
wb.record("19.1a", 0.2 * 0.9**3, decimals=4, mistakes={"à l'epoch 0, η vaut η0 : combien de multiplications à l'epoch 3 ?": 0.2 * 0.9**4,
          "à l'epoch 3, η a été multiplié 3 fois": 0.2 * 0.9**2, "n'oublie pas η0 = 0,2": 0.9**3})
wb.record("19.1b", 0.2 * 0.9**10, decimals=4, mistakes={"à l'epoch 10, η a été multiplié 10 fois": 0.2 * 0.9**9,
          "10 multiplications, pas 11": 0.2 * 0.9**11})
wb.record("19.1c", 0.2 * 0.5**(3 // 4), decimals=3, mistakes={"⌊3/4⌋ vaut combien ?": 0.1})
wb.record("19.1d", 0.2 * 0.5**(4 // 4), decimals=3, mistakes={"que vaut ⌊4/4⌋ ?": 0.2,
          "⌊4/4⌋ vaut combien ?": 0.05})
wb.record("19.1e", 0.2 * 0.5**(11 // 4), decimals=3, mistakes={"⌊11/4⌋ vaut combien ?": 0.025, "compte les paliers déjà franchis à l'epoch 11": 0.1})
wb.record("19.1f", 0.5**(1 / 4), decimals=4, mistakes={"le planning multiplie : il faut γ⁴ = 0,5, pas une baisse de 12,5 % par epoch": 0.875,
          "une division par 2 toutes les 4 epochs, pas toutes les 2": 0.5**0.5,
          "γ s'applique à chaque epoch : sa puissance 4 doit valoir 0,5": 0.5})
wb.record("19.1g", 0.2 * 0.9**(8 - 4), decimals=4, mistakes={"l'exposant compte les epochs depuis la fin du palier": 0.2 * 0.9**8,
          "k − 4 pour k = 8 : combien d'epochs après le palier ?": 0.2 * 0.9**3,
          "l'exposant est k − 4, pas k − 3": 0.2 * 0.9**5})

WB_ANSWER {"decimals": 4, "hash": "024139939647df9997d73166a32d5f8454a68cdf1920656bbde1220511a6939b", "hash_coarse": "217165ba28d47d3494a60e987650c8e3c8efe47525069256d2cb4e9beee167cc", "id": "19.1a", "kind": "float", "magnitude": -1, "mistakes": {"6dfdc17e3c1e496ebd77081193c48cfd8e099e37c99f7625520e217637799098": "à l'epoch 0, η vaut η0 : combien de multiplications à l'epoch 3 ?", "c33338591c676ede4491e4d0571af8f85f217160ec343e3bf0d2e7a81f5ec001": "à l'epoch 3, η a été multiplié 3 fois", "e6daa56ab3ef36eac45f3b5f27c884c0d76b5a186a3e0f991f84a203a22f85b0": "n'oublie pas η0 = 0,2"}, "sign": 1}
📝 Ex 19.1a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "d657cd9242dfde2fa8dd9492e3a70662b2172420a6093f47ea9973010a681eec", "hash_coarse": "364be80a464b5ff30cc81f02090ea05f0b31e2931704bd68692e8687a97e1903", "id": "19.1b", "kind": "float", "magnitude": -2, "mistakes": {"185123999b6684864a90f9ae2909d914c034e0fb3207de08f51ecd4f80fe0fca": "10 multiplications, pas 11", 

**Ex 19.2 — Trois pas de momentum, deux conventions**

In [19]:
wb.record("19.2a", 2 * 1.0, decimals=2, mistakes={"b1 = g1 : ni η ni (1 − μ) n'y entrent": 0.2, "le gradient de L(w) = w² est 2w : où l'évalues-tu ?": 1.0})
wb.record("19.2b", momentum_torch_192([0.1]), decimals=2, mistakes={"w1 = w0 − η b1, avec b1 = g1": 0.98, "on descend : w − η b": 1.2})
wb.record("19.2c", 0.9 * 2 + 2 * momentum_torch_192([0.1]), decimals=2, mistakes={"g2 se calcule au point w1, pas en w0": 3.8, "b2 = μ b1 + g2 : n'oublie pas μ b1": 1.6})
wb.record("19.2d", momentum_torch_192([0.1, 0.1]), decimals=2, mistakes={"le pas utilise la vitesse b2, pas le gradient g2": 0.64,
          "g2 se calcule au point w1": 0.42,
          "w2 = w1 − η b2 : pars de w1": 0.54})
wb.record("19.2e", momentum_torch_192([0.1, 0.1, 0.1]), decimals=3, mistakes={"le pas utilise la vitesse b3, pas le gradient g3": 0.368})
wb.record("19.2f", abs(momentum_book_192([0.1, 0.1]) - momentum_torch_192([0.1, 0.1])) < 1e-12, mistakes={"déroule Δ1 = −η g1 et Δ2 = γ Δ1 − η g2, puis compare": False})
wb.record("19.2g", momentum_torch_192([0.1, 0.05]), decimals=2, mistakes={"relis où intervient η dans cette convention-ci": 0.54,
          "le learning rate a changé au deuxième pas": 0.46})
wb.record("19.2h", momentum_book_192([0.1, 0.05]), decimals=2, mistakes={"relis où intervient η dans cette convention-là : Δ1 garde le η de son pas": 0.63,
          "le learning rate a changé au deuxième pas, dans Δ2": 0.46})

WB_ANSWER {"decimals": 2, "hash": "0d94bacfaeabfa50a833f549c4de36786fb0bf6aa4686de21d6b8462b366148b", "hash_coarse": "7cb22dadaf16dcb61204a2fbb1bd8296a7ed81eebb150f3521d31c7858fa317b", "id": "19.2a", "kind": "float", "magnitude": 0, "mistakes": {"822b1019ad953445570610a8ac2392b29df19cab9a064e9ee5b620a1795026a7": "le gradient de L(w) = w² est 2w : où l'évalues-tu ?", "dd91a1e9270ff2903b749fd7912172e5bce997e46504b89f20110a76db640550": "b1 = g1 : ni η ni (1 − μ) n'y entrent"}, "sign": 1}
📝 Ex 19.2a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "3889b775f3ee6843b7c68b9316a127acc114448adeecef902015cf1029090513", "hash_coarse": "4cef8a20c06b5c90290224dadfaed7a8333660552ed8fbe980b5a93ea69baccc", "id": "19.2b", "kind": "float", "magnitude": -1, "mistakes": {"2015394f01550f7d3aa225fb8c6c4fdb6c0ab4fb83cbb7201387ac47e86f1ea5": "on descend : w − η b", "344c769d05290464c6a09cea14822c1d2de6be8dcb57e389599b80fea14a12b1": "w1 = w0 − η b1, avec b1 = g1"}, "sign": 1}
📝 

**Ex 19.3 — Nesterov : gradient anticipé et reformulation de PyTorch**

In [20]:
wb.record("19.3a", nesterov_book_193(2)[1][0], decimals=3, mistakes={"le point anticipé ajoute μ v1 à w1": 0.8, "v1 est négatif : attention au signe": 0.98})
wb.record("19.3b", nesterov_book_193(2)[1][1], decimals=3, mistakes={"le gradient se prend au point anticipé, pas en w1": 1.6,
          "v1 est négatif : refais le point anticipé": 1.96})
wb.record("19.3c", nesterov_book_193(2)[1][2], decimals=3, mistakes={"le gradient qui entre dans v2 se prend au point anticipé": 0.46,
          "v2 = μ v1 − η g : n'oublie pas μ v1": 0.676})
wb.record("19.3d", nesterov_book_193(3)[2][2], decimals=4, mistakes={"à chaque pas, le gradient se prend au point anticipé": 0.062})
wb.record("19.3e", nesterov_torch_193(1)[0], decimals=3, mistakes={"le pas de PyTorch est η (g + μ b), pas η b": 0.8,
          "p1 = p0 − η (g1 + μ b1) : pars de p0 = 1": 0.38})
wb.record("19.3f", nesterov_torch_193(2)[1], decimals=4, mistakes={"le pas de PyTorch combine g2 et μ b2": 0.496,
          "le pas utilise g + μ b, pas b seul": 0.316})
wb.record("19.3g", "B", mistakes={"compare p2 à w2 : sont-ils égaux ?": "A", "compare p2 à w3": "C",
          "v2 est une vitesse, p2 une position": "D"})

WB_ANSWER {"decimals": 3, "hash": "b13972248652595593005b3ab2789084d45c3f3db0d25b4a3ceead50a522489f", "hash_coarse": "659bd2781da9036a01406e33687b0bc6b75884a4085c52197a9c8a295f59320c", "id": "19.3a", "kind": "float", "magnitude": -1, "mistakes": {"0b44a64b9989d902c339b189cf30b31f16083288d94e93e2d14bc093d6e77001": "le point anticipé ajoute μ v1 à w1", "97c4f839f21b1c5f7d39ae224978b74dafdf547f855752b9c0faa16c768792b0": "v1 est négatif : attention au signe"}, "sign": 1}
📝 Ex 19.3a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "c03badca167f83c7fec2e8f98022766ff8b25667e8dc4222e74f9dc5bd9648df", "hash_coarse": "f21050071136942493d5f9bfd351447e1873c53d27ba40f22a5161b7041a1e71", "id": "19.3b", "kind": "float", "magnitude": 0, "mistakes": {"3b7227c5c1109b77f986e5369cbf9d519d525fa01be07f0fe2109b7e4e6bf821": "le gradient se prend au point anticipé, pas en w1", "453bae0ece822ae19076eec854464efc389b163aa5d1d93d08284b720f7af44f": "v1 est négatif : refais le point an

**Ex 19.4 — Adagrad et RMSprop sur une suite de gradients**

In [21]:
wb.record("19.4a", adagrad_194()[1][0], decimals=1, mistakes={"s2 = s1 + g2² : le carré de −1 est positif": 8.0,
          "s accumule TOUS les carrés depuis le début": 1.0})
wb.record("19.4b", adagrad_194()[1][1], decimals=4, mistakes={"g2 est négatif : le pas fait monter w": -0.1 - 0.1 / math.sqrt(10),
          "divise g2 par la racine de s2, pas de s1": -0.1 + 0.1 / 3})
wb.record("19.4c", adagrad_194()[2][1], decimals=4, mistakes={"s3 = s2 + g3² : le carré de 2": adagrad_194()[1][1] - 0.2 / math.sqrt(12),
          "le troisième pas se divise par la racine de s3, pas de s2": adagrad_194()[1][1] - 0.2 / math.sqrt(10)})
wb.record("19.4d", rmsprop_194()[0][0], decimals=2, mistakes={"le coefficient du nouveau carré est 1 − α": 8.1, "c'est le CARRÉ du gradient qui entre dans v": 0.3})
wb.record("19.4e", rmsprop_194()[0][1], decimals=4, mistakes={"divise par la racine de v1, celle de d)": -0.1,
          "divise par la racine de v1, pas par v1": -0.1 * 3 / 0.9})
wb.record("19.4f", rmsprop_194()[2][0], decimals=3, mistakes={"continue jusqu'au troisième pas": 0.91,
          "v3 = α v2 + (1 − α) g3²": 0.91 + 0.4})
wb.record("19.4g", rmsprop_194()[2][1], decimals=4, mistakes={"continue jusqu'au troisième pas : w3": rmsprop_194()[1][1]})
wb.record("19.4h", 1 / math.sqrt(1 - 0.9), decimals=4, mistakes={"l'inverse : le premier pas de RMSprop est-il plus grand ou plus petit ?": math.sqrt(0.1),
          "il y a une racine : le rapport des racines": 10.0})
wb.record("19.4i", 1 / math.sqrt(100), decimals=4, mistakes={"Adagrad divise par la RACINE de la somme": 0.01, "l'inverse : le pas grandit-il ?": 10.0,
          "compte les carrés que contient la somme au 100ᵉ pas": 1 / math.sqrt(99),
          "au 100ᵉ pas, combien de carrés la somme contient-elle, exactement ?": 1 / math.sqrt(101)})

WB_ANSWER {"decimals": 1, "hash": "d245c3c91c627e3ea34428112512109a2d3b1a7832a1c0dc5a142b165290e085", "hash_coarse": "98288a78191fd57fe6f7a197f9c979c2ceda42c8c86b7aef2514c85f0561b435", "id": "19.4a", "kind": "float", "magnitude": 1, "mistakes": {"b18fcc2bec98bea34cc4b2bc7897a871c722178390771f2432340b64621387a4": "s accumule TOUS les carrés depuis le début", "f67bacd5dec1672c6e4dc81bea7b8474677a0b235a1708934b7e28d1970a20a2": "s2 = s1 + g2² : le carré de −1 est positif"}, "sign": 1}
📝 Ex 19.4a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "fabe5a4ef5cbbe6eddf882b8668f8f6a23ee8ed29fc04b559c58a81a0a646d61", "hash_coarse": "81c1505a638a7afd88a742776611686cdea403db655e2cd11b7a4080383c23cd", "id": "19.4b", "kind": "float", "magnitude": -2, "mistakes": {"3f58970b151ce2b83d608530ea645327fecae7138c929cad5861d6a6c65405b0": "divise g2 par la racine de s2, pas de s1", "4830a09b2f6fff0f4ba9d705d764729bd393c46ed3c276ebdfbf8395745e5214": "g2 est négatif : le pas fait 

**Ex 19.5 — Un pas d'Adam et la correction de biais**

In [22]:
wb.record("19.5a", adam_195([0.5])[0][0], decimals=3, mistakes={"m1 = β1 m0 + (1 − β1) g1, avec m0 = 0": 0.45, "m1 ou m̂1 ? relis ce qui est demandé": 0.5})
wb.record("19.5b", adam_195([0.5])[0][1], decimals=6, mistakes={"c'est le CARRÉ du gradient qui entre dans v": 0.0005, "c'est v̂1 : on demande v1": 0.25})
wb.record("19.5c", adam_195([0.5])[0][2], decimals=3, mistakes={"m̂1 ou m1 ? relis ce qui est demandé, et la correction": 0.05, "divise par 1 − β1, pas par β1": 0.05 / 0.9})
wb.record("19.5d", adam_195([0.5])[0][4], decimals=3, mistakes={"on descend : w − η m̂ / √v̂": 1.01, "le pas d'Adam est η m̂ / √v̂, pas η g": 0.995})
wb.record("19.5e", 0.01 * 0.05 / math.sqrt(0.00025), decimals=4, mistakes={"ici, sans correction de biais : utilise m1 et v1 tels quels": 0.01,
          "divise par la RACINE de v1": 0.01 * 0.05 / 0.00025})
wb.record("19.5f", adam_195([0.5, -0.1])[1][0], decimals=3, mistakes={"g2 est négatif": 0.055, "m2 = β1 m1 + (1 − β1) g2 : n'oublie pas β1 devant m1": 0.04})
wb.record("19.5g", adam_195([0.5, -0.1])[1][2], decimals=4, mistakes={"corrige m2 avec 1 − β1²": 0.035, "au deuxième pas, t = 2 : 1 − β1², pas 1 − β1": 0.35})
wb.record("19.5h", adam_195([0.5, -0.1])[1][4], decimals=5, mistakes={"le deuxième pas d'Adam vaut-il encore η exactement ?": 0.98,
          "regarde le signe de m̂2 : w monte-t-il ou baisse-t-il ?": 0.99 + (0.99 - adam_195([0.5, -0.1])[1][4])})
wb.record("19.5i", adam_195([0.5, -0.1])[1][4] < adam_195([0.5])[0][4], mistakes={"regarde le signe de m̂2": False})
wb.record("19.5j", abs(adam_195([500.0, -100.0])[1][4] - adam_195([0.5, -0.1])[1][4]) < 1e-12, mistakes={"multiplie tous les g par 1 000 : que deviennent m et √v ?": False})

WB_ANSWER {"decimals": 3, "hash": "974f9a30efe8e49546307eee983cc8d01ec69b4b6f9f72771639ad92fc2e33bd", "hash_coarse": "1a299a198c7b4205161a8cbc1f39e62d814df2437b08956dbfd7fe5508aadcae", "id": "19.5a", "kind": "float", "magnitude": -2, "mistakes": {"b12625c8c4c1414e9ecb39c59b97506757afe532750e6b9055edb1aa598f8825": "m1 ou m̂1 ? relis ce qui est demandé", "fd1eb88c1bfdec57f9d13a600aa64990d540cf0e71d57b11bc8700431b528607": "m1 = β1 m0 + (1 − β1) g1, avec m0 = 0"}, "sign": 1}
📝 Ex 19.5a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 6, "hash": "6877625074e63120795013aea8ae079d20bca681b96f0f3b739e831ce1d7da4c", "hash_coarse": "a05c3d019938d03fd3c774c5ac0b978fc13d930d56658b42234843c2613df62c", "id": "19.5b", "kind": "float", "magnitude": -4, "mistakes": {"a9854c89a55e9e7130f1fccd26606c7c1e16695776b4edcd2fdff4f881484547": "c'est le CARRÉ du gradient qui entre dans v", "ea29fb574c251df9d7d7d9759b421e80faa1e8726ada562ea16ec66e4b9487ca": "c'est v̂1 : on demande v1"}, "sign":

**Ex 19.6 — Adam + L2 ou AdamW : la même pénalité ? Convertir alpha, lambda et weight_decay**

In [23]:
wb.record("19.6a", 0.01, decimals=3, mistakes={"dérive (λ/2)‖w‖² : le ½ disparaît": 0.02, "dérive (λ/2)‖w‖² : pas de ½ en plus": 0.005})
wb.record("19.6b", 2 * 0.01, decimals=3, mistakes={"sans le ½, la dérivée de λ‖w‖² fait apparaître un 2": 0.01,
          "sans le ½ : la dérivée multiplie, elle ne divise pas": 0.005})
wb.record("19.6c", 2 - 0.1 * 0.01 * 2, decimals=3, mistakes={"le weight decay est multiplié par le learning rate": 1.98,
          "le terme ajouté au gradient est wd × w, avec w = 2": 1.999})
wb.record("19.6d", 1 - 1e-3 * 1e-2, decimals=5, mistakes={"le facteur est 1 − lr × wd : multiplie wd par lr": 0.99,
          "le facteur est 1 − lr × wd, pas 1 − lr": 0.999})
wb.record("19.6e", (1 - 1e-3 * 1e-2) ** 10_000, decimals=4, mistakes={"c'est une puissance du facteur, pas une soustraction répétée": 0.9,
          "10 000 pas : la puissance est 10 000": (1 - 1e-5) ** 1000})
wb.record("19.6f", 2 - 1e-3, decimals=3, mistakes={"relis la convention de Adam(weight_decay=...) en tête de l'exercice": 2 * (1 - 1e-5),
          "le premier pas d'Adam a une taille fixe : laquelle ?": 1.98})
wb.record("19.6g", 2 * (1 - 1e-3 * 1e-2), decimals=5, mistakes={"relis la convention d'AdamW en tête de l'exercice": 1.999,
          "la décroissance est multipliée par le learning rate": 1.98})
wb.record("19.6h", 1e-4 / 1e-3, decimals=2, mistakes={"PyTorch multiplie le weight_decay par le learning rate : il faut compenser": 0.0001,
          "rapport inversé : il faut lr × weight_decay = λ": 10.0})
wb.record("19.6i", 1e-4 / 200 / 1e-7, decimals=1, mistakes={"divise alpha par la taille du mini-batch": 1000.0,
          "en multiple de 10⁻⁷ : compte bien les zéros": 0.5,
          "en multiple de 10⁻⁷ : écris le coefficient devant 10⁻⁷, pas la valeur": 5e-7,
          "divise alpha par la taille du mini-batch, ne multiplie pas": 0.05,
          "compte bien les puissances de 10 de alpha et de la taille du mini-batch": 500.0})
wb.record("19.6j", False, mistakes={"avec momentum, le terme wd × w couplé entre dans la vitesse b : écris les deux versions sur deux pas": True})

WB_ANSWER {"decimals": 3, "hash": "342d6de39e6dec69c0a1edaec49e9f0bad7afd23fdeea4d63472fe0e9accc045", "hash_coarse": "afb026432f2dda012debf8a82b4f53f566e64243550977be3993a3c71308c0e2", "id": "19.6a", "kind": "float", "magnitude": -2, "mistakes": {"5423a4ada1d6c845516a12a1fbd581cab9c99af6eb4e77ed774ff76c970b45b6": "dérive (λ/2)‖w‖² : pas de ½ en plus", "59becc0ad6318793e59091acc9ec1b158c4d9efefa84197e8bfcf3b3fdb6cc64": "dérive (λ/2)‖w‖² : le ½ disparaît"}, "sign": 1}
📝 Ex 19.6a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "00b2069070fc24f811851650e74015c72397a8aa93ed981a985c38e58481c191", "hash_coarse": "5d2d30b0173dcb707658e6614da5894a45bce1ceebf4c963798d7f2753cc2ff6", "id": "19.6b", "kind": "float", "magnitude": -2, "mistakes": {"008856eba6d93027d015acb11316d00e47fd7571ef5fa02ace10ea02d2d3cb13": "sans le ½ : la dérivée multiplie, elle ne divise pas", "65c5d06eaacabc9177a9cf71e623c2b9116023b5624312526751c0b7a024bacb": "sans le ½, la dérivée de λ‖w‖² f

**Ex 19.9 — Fermi : la mémoire des optimiseurs, du MLP au LLM**

In [24]:
wb.record("19.9a", P_199, mistakes={"les biais sont des paramètres aussi": W_199, "compte aussi la dernière couche, 256 × 10 + 10": P_199 - 2570})
wb.record("19.9b", 4 * P_199 / 1e6, decimals=1, mistakes={"4 octets par nombre float32": P_199 / 1e6, "32 bits = 4 octets : compte en octets": 32 * P_199 / 1e6,
          "float32 = 4 octets, pas 2": 2 * P_199 / 1e6})
wb.record("19.9c", 2 * 4 * P_199 / 1e6, decimals=1, mistakes={"Adam garde deux nombres par paramètre": 4 * P_199 / 1e6,
          "les états seulement, sans les paramètres ni les gradients": 4 * 4 * P_199 / 1e6})
wb.record("19.9d", 4 * 4 * P_199 / 1e6, decimals=1, mistakes={"paramètres, gradients ET deux états": 3 * 4 * P_199 / 1e6,
          "il faut aussi les paramètres et les gradients": 2 * 4 * P_199 / 1e6})
wb.record("19.9e", round(7e9 * (2 + 2 + 4 + 4 + 4) / 1e9), fractional="un nombre de Go : un entier",
          mistakes={"compte bien chaque poste : la copie maîtresse float32 et les deux tampons d'Adam, m et v": 84,
          "additionne tous les octets par paramètre": 56})
wb.record("19.9f", round(7e9 * (2 + 2 + 4 + 4) / 1e9), fractional="un nombre de Go : un entier",
          mistakes={"SGD avec momentum garde UN tampon": 112, "n'oublie pas la copie maîtresse float32": 56})
wb.record("19.9g", math.ceil(112 / 80), fractional="un nombre de GPU : un entier, arrondi au-dessus",
          mistakes={"les états de e) tiennent-ils dans un seul GPU de 80 Go ?": 1})

WB_ANSWER {"hash": "b9949dbbf28ff3192a65834d7f0e7ee3b91b67f429ff39b54f1c3c2a711119a9", "id": "19.9a", "kind": "int", "magnitude": 6, "mistakes": {"bd4516eeef63f4ec83be2a3f362a09fdaac254c00eddd8ffa5cf1325fb63c0f4": "les biais sont des paramètres aussi", "d4cb4186f01098837ace88e958a921adc0e8d5dd28250df828d9a2d68d9ab568": "compte aussi la dernière couche, 256 × 10 + 10"}, "sign": 1}
📝 Ex 19.9a : réponse enregistrée (int).
WB_ANSWER {"decimals": 1, "hash": "c5774e0411d174b7e40e381513cc382aab9705183f775bd03d71fcebcab0a6f1", "hash_coarse": "302e1c9965014a639346bb042bd5a24d4ea6fcf5a897a3f3f96acc81fb820c49", "id": "19.9b", "kind": "float", "magnitude": 0, "mistakes": {"3b163f11ac8e089a42aedda51ebddac0ac485f805eea2b3358aa321d9e6dad61": "float32 = 4 octets, pas 2", "4970d2a1d5c9afaddb0059ae4bbf76e27a8fc0d6898b7f1d9db354425f57753c": "32 bits = 4 octets : compte en octets", "533649918a4e4741f5fc6a719cd5972ed31f7f566fc30585d2923753510a4a66": "4 octets par nombre float32"}, "sign": 1}
📝 Ex 19.9b : r

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dérouler à la main deux pas de momentum, de RMSprop et d'Adam, correction de biais comprise ?
2. Sais-tu expliquer pourquoi la courbure limite le learning rate, et ce que change une vallée étroite ?
3. Sais-tu dire la différence entre Adam avec `weight_decay` et AdamW, et ce que font le warmup, le cosinus et le clipping ?

**Pour aller plus loin** : la synthèse de S. Ruder et l'article interactif de G. Goh sur le momentum, cités dans la fiche. La suite : le ch. 20, où PyTorch fera tout cela avec `torch.optim`.

*Les exercices de code 19.14 à 19.28 (parties A à D, ta librairie `optim.py`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 19` pour obtenir le notebook complet.*